# Step By Step 06 - 完整的回测

Run a Parquet-to-factor-to-execution workflow with stored backtest results.


In [ ]:
import asyncio
import datetime as dt
import numpy as np
import pyarrow as pa
import pyarrow.compute as pc

rng = np.random.default_rng(42)
n_dates = 6
n_assets = 12
dates = [dt.datetime(2026, 1, 5) + dt.timedelta(days=index) for index in range(n_dates)]
codes = np.array([f"{index:06d}" for index in range(1, n_assets + 1)])
n_rows = n_dates * n_assets
panel = pa.table({
    "trade_date": [value for value in dates for _ in range(n_assets)],
    "code": np.tile(codes, n_dates),
    "factor": rng.normal(size=n_rows),
    "value": rng.normal(size=n_rows),
    "dx": rng.normal(0.0, 0.01, size=n_rows),
    "industry": np.tile(np.array(["Bank", "Auto", "Health"]), n_dates * n_assets // 3),
    "benchmark_weight": np.full(n_rows, 1.0 / n_assets),
    "BETA": np.tile(np.linspace(0.8, 1.2, n_assets) ** 2, n_dates),
    "SIZE": np.tile(np.linspace(-1.0, 1.0, n_assets), n_dates),
})
import tempfile
from alphamind.data import read_parquet, write_parquet
from alphamind.data.processing import factor_processing
from alphamind.data.standardize import standardize
from alphamind.data.transformer import Transformer
from alphamind.data.winsorize import winsorize_normal
from alphamind.execution.naiveexecutor import NaiveExecutor
from alphamind.formula import CSRank
from alphamind.portfolio.rankbuilder import rank_build

with tempfile.TemporaryDirectory() as directory:
    input_path = f"{directory}/input.parquet"
    output_path = f"{directory}/returns.parquet"
    write_parquet(panel, input_path)
    loaded = read_parquet(input_path)
    scores = await asyncio.to_thread(
        Transformer({"score": CSRank("factor")}).transform, "code", loaded
    )
    loaded = loaded.append_column("score", scores["score"])
    executor = NaiveExecutor()
    results = []
    loaded_dates = sorted(pc.unique(loaded["trade_date"]).to_pylist())
    for trade_date in loaded_dates:
        section = loaded.filter(pc.equal(loaded["trade_date"], trade_date))
        signal = section["score"].to_numpy(zero_copy_only=False).reshape(-1, 1)
        processed = factor_processing(
            signal, pre_process=[winsorize_normal, standardize], post_process=[standardize]
        ).flatten()
        weights = rank_build(processed, use_rank=3).flatten().astype(float) / 3.0
        target = pa.table({"code": section["code"], "weight": weights})
        turn_over, executed = executor.execute(target)
        executor.set_current(executed)
        portfolio_return = float(weights @ section["dx"].to_numpy(zero_copy_only=False))
        results.append(pa.table({
            "trade_date": [trade_date],
            "return": [portfolio_return],
            "turn_over": [turn_over],
        }))
    backtest = pa.concat_tables(results)
    write_parquet(backtest, output_path)
    restored = read_parquet(output_path)
assert restored.num_rows == n_dates
restored
